In [ ]:
# List of required packages
required_packages <- c( 
    "R.utils",
    "dplyr", 
    "tidyr",
    "data.table",
    "ggplot2",
    "lubridate",
    "broom",
    "scales",
    "patchwork",
    "glmnet"
)

# Function to check and install missing packages
install_and_load <- function(packages) {
  installed_packages <- rownames(installed.packages())
  for (pkg in packages) {
    if (!pkg %in% installed_packages) {
      install.packages(pkg, dependencies = TRUE)
    }
    suppressMessages(library(pkg, character.only = TRUE))
  }
}

# Install and load the required packages
install_and_load(required_packages)

In [ ]:
# Define the file name
file_names <- c(
    "ukb_gp_for_analysis.rds"
)

# Check if the file exists in the current directory
for (file in file_names) {
    if (!file.exists(file)) {
      message("File not found. Downloading from DNAnexus...")

      # Execute the dx download command
      system(paste("dx download project-XXXXXXXXXXXXXXXXXXXXXXXX:/path/to/scratch/", file, sep = ""))
    }
}

# Load the data
data_raw <- readRDS("ukb_gp_for_analysis.rds")

In [ ]:
y_var   <- "LDL1"

log_vars    <- c("LDL0")  # log() in model
num_vars    <- c("age_at_LDL0", "n_rx_days_365d", "n_clin_days_365d", "ldl_prs")
factor_vars <- c("sex", "prior_statin", "prior_CAD")

# ---- helper: complete-case restriction respecting log() constraints ----
prep_complete_cases <- function(dt, y, log_vars, num_vars, factor_vars, log_y = FALSE) {
  vars_needed <- unique(c(y, log_vars, num_vars, factor_vars))
  keep <- stats::complete.cases(dt[, ..vars_needed])

  # log() covariates require strictly positive values
  for (v in log_vars) keep <- keep & is.finite(dt[[v]]) & dt[[v]] > 0

  # outcome constraints depend on whether it's logged
  keep <- keep & is.finite(dt[[y]])
  if (isTRUE(log_y)) keep <- keep & dt[[y]] > 0

  dt[keep]
}

data <- prep_complete_cases(data_raw, y_var, log_vars, num_vars, factor_vars, log_y = TRUE)

In [ ]:
# ---- build formula programmatically ----
log_terms    <- sprintf("I(log(%s))", log_vars)
num_terms    <- num_vars
factor_terms <- sprintf("factor(%s)", factor_vars)

In [ ]:
data1 <- data %>% filter(A == 1)

# outcomes
data1 <- data1 %>%
mutate(
  abs_change = LDL1 - LDL0,
  log_ratio  = log(LDL1) - log(LDL0)   # = log(Y/LDL0)
)

In [ ]:
rhs_terms <- c(num_terms, factor_terms, "ldl_prs")
m_abs   <- lm(as.formula(paste0("abs_change ~ ", paste(rhs_terms, collapse = " + "))), data = data1)
m_rel   <- lm(as.formula(paste0("log_ratio ~ ", paste(rhs_terms, collapse = " + "))), data = data1)

rhs_terms <- c(num_terms, factor_terms, "I(log(LDL0))", "ldl_prs")
m_rel_adj <- lm(as.formula(paste0("log_ratio ~ ", paste(rhs_terms, collapse = " + "))), data = data1)

In [ ]:
summary(m_rel)

In [ ]:
beta_ukb <- summary(m_rel)$coefficients["ldl_prs", "Estimate"]
se_ukb <- summary(m_rel)$coefficients["ldl_prs", "Std. Error"]
beta_hrn <- -0.011494 # From Liz
se_hrn <- (1/qnorm(0.975)) * (beta_hrn - -0.021158) # From Liz
beta_ivw <- (beta_ukb/se_ukb^2 + beta_hrn/se_hrn^2)/(1/se_ukb^2 + 1/se_hrn^2)
se_ivw <- sqrt(1/(1/se_ukb^2 + 1/se_hrn^2))
pval_ivw <- 2 * pnorm(abs(beta_ivw / se_ivw), lower.tail=FALSE)
cil_ivw <- beta_ivw - qnorm(0.975) * se_ivw
cih_ivw <- beta_ivw + qnorm(0.975) * se_ivw
print(beta_ivw)
print(cil_ivw)
print(cih_ivw)
print(pval_ivw)

In [ ]:
model_list <- list(m_abs, m_rel, m_rel_adj)
out_list <- lapply(X = model_list, FUN = function(m) {
    data.frame(
      estimate = coef(m)[["ldl_prs"]],
      se = summary(m)$coefficients["ldl_prs", "Std. Error"],
      lower_ci = confint(m)["ldl_prs", 1],
      upper_ci = confint(m)["ldl_prs", 2]
  )
})
out_change <- do.call(rbind, out_list)
out_change$estimator <- c("Absolute change in LDL-C",
  "Unadjusted relative change in LDL-C",
  "Adjusted relative change in LDL-C"
)

In [ ]:
plot_dat_change <- out_change %>%
  mutate(
    estimator = factor(
      estimator,
      levels = c(
        "Absolute change in LDL-C",
        "Unadjusted relative change in LDL-C",
        "Adjusted relative change in LDL-C"
      )
    ),
    label = sprintf(
      "est=%.3f\n95%% CI\n[%.3f, %.3f]",
      estimate, lower_ci, upper_ci
    )
  ) %>%
  arrange(estimator)

# Reuse one pink colour for all three, distinguished by alpha
cols_change <- c(
  "Absolute change in LDL-C" = "#E7298A",
  "Unadjusted relative change in LDL-C" = "#E7298A",
  "Adjusted relative change in LDL-C" = "#E7298A"
)

alpha_change <- c(
  "Absolute change in LDL-C" = 0.75,
  "Unadjusted relative change in LDL-C" = 0.75,
  "Adjusted relative change in LDL-C" = 0.75
)

y_rng_c <- range(c(plot_dat_change$lower_ci, plot_dat_change$upper_ci), na.rm = TRUE)
y_lab_c <- max(plot_dat_change$upper_ci, na.rm = TRUE) + 0.10 * diff(y_rng_c)

plot_dat_change <- plot_dat_change %>%
  mutate(y = y_lab_c)

theme_pub <- theme_minimal(base_size = 12) +
  theme(
    panel.grid.minor = element_blank(),
    panel.grid.major.y = element_blank(),
    legend.title = element_blank(),
    legend.position = "top",
    plot.title.position = "plot",
    plot.title = element_text(face = "bold"),
    plot.subtitle = element_text(colour = "grey25"),
    axis.title = element_text(face = "bold")
  )

p_box_change <- ggplot(
  plot_dat_change,
  aes(x = estimator, y = estimate, colour = estimator)
) +
  geom_hline(yintercept = 0, linetype = "dotted", linewidth = 0.8) +
  geom_errorbar(
    aes(ymin = lower_ci, ymax = upper_ci, alpha = estimator),
    width = 0.18,
    linewidth = 0.9,
    show.legend = FALSE
  ) +
  geom_point(
    aes(fill = estimator, alpha = estimator),
    size = 3.2,
    shape = 21,
    stroke = 0.8,
    colour = "black",
    show.legend = FALSE
  ) +
  geom_text(
    aes(y = y, label = label),
    colour = "black",
    size = 3.2,
    lineheight = 1.05,
    show.legend = FALSE
  ) +
  coord_cartesian(clip = "off") +
  scale_fill_manual(values = cols_change) +
  scale_colour_manual(values = cols_change) +
  scale_alpha_manual(values = alpha_change) +
  theme_pub +
  theme(
    legend.position = "none",
    axis.title.x = element_blank(),
    axis.text.x = element_text(angle = 18, hjust = 1),
    plot.margin = margin(8, 12, 8, 12)
  ) +
  labs(
    title = "Initiator-only designs in statin users",
    # subtitle = "Frequentist point estimates with 95% confidence intervals.",
    y = "Estimated coefficient on LDL PRS"
  )

p_box_change
ggsave(
  plot = p_box_change,
  width = 8,
  height = 6,
  dpi = 300,     
  filename = "statin_ldl_change_score_forest.png"
)